In [1]:
!pip install -U \
langchain \
langchain-community \
langchain-groq \
langchain-huggingface \
langchain-text-splitters \
faiss-cpu \
sentence-transformers \
unstructured \
python-dotenv \
tiktoken

In [4]:
import langchain
print(langchain.__version__)

1.3.14


In [5]:
import os
import pickle

from google.colab import userdata

from langchain_groq import ChatGroq
from langchain_huggingface import HuggingFaceEmbeddings

from langchain_community.document_loaders import UnstructuredURLLoader
from langchain_community.vectorstores import FAISS

from langchain_text_splitters import RecursiveCharacterTextSplitter

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

In [6]:
os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")

In [7]:
from google.colab import userdata
import os

os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")

In [8]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    model="llama-3.1-8b-instant",
    temperature=0
)

response = llm.invoke("Hello")

print(response.content)

Hello. How can I assist you today?


In [9]:
loader = UnstructuredURLLoader(
    urls=[
        "https://www.moneycontrol.com/news/business/markets/wall-street-rises-as-tesla-soars-on-ai-optimism-11351111.html",
        "https://www.moneycontrol.com/news/business/tata-motors-launches-punch-icng-price-starts-at-rs-7-1-lakh-11098751.html",
    ]
)

documents = loader.load()

In [10]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

docs = splitter.split_documents(documents)

In [11]:
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-mpnet-base-v2"
)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/11.6k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [12]:
vectorstore = FAISS.from_documents(
    docs,
    embeddings
)

In [13]:
retriever = vectorstore.as_retriever(
    search_kwargs={"k":3}
)

In [14]:
llm = ChatGroq(
    model="llama-3.1-8b-instant",
    temperature=0
)

In [26]:
prompt = ChatPromptTemplate.from_template("""
You are an Equity Research Assistant.

Use ONLY the provided context to answer the question.

If the answer cannot be found in the context, say:
"I couldn't find the answer in the provided documents."

Context:
{context}

Question:
{question}
""")

In [15]:
prompt = ChatPromptTemplate.from_template(
"""
You are an equity research assistant.

Answer ONLY using the context below.

Context:
{context}

Question:
{question}
"""
)

In [27]:
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

In [28]:
rag_chain = (
    {
        "context": retriever | format_docs,
        "question": RunnablePassthrough(),
    }
    | prompt
    | llm
    | StrOutputParser()
)

In [29]:
query = "What is the price of Tiago iCNG?"


In [30]:
answer = rag_chain.invoke(query)

In [31]:
print(answer)

The price of Tiago iCNG is between Rs 6.55 lakh and Rs 8.1 lakh.


In [32]:
query = "What are the main features of Tata Punch iCNG?"
answer = rag_chain.invoke(query)
answer

"The main features of Tata Punch iCNG include:\n\n1. The company's proprietary twin-cylinder technology\n2. Enhanced safety features like a micro-switch to keep the car switched off at the time of refuelling and thermal incident protection that cuts off CNG supply to the engine and releases gas into the atmosphere\n3. Voice assisted electric sunroof\n4. Automatic projector headlamps\n5. LED DRLs\n6. 16-inch diamond cut alloy wheels\n7. 7-inch infotainment system by Harman that supports Android Auto and Apple Carplay connectivity\n8. Rain sensing wipers\n9. Height adjustable driver seat"

In [23]:
query = "Why did Tesla shares rise?"
answer = rag_chain.invoke(query)
answer

'Tesla shares rose 10% after Morgan Stanley upgraded the electric car maker to "overweight" from "equal-weight," citing its Dojo supercomputer, which could boost the company\'s market value by nearly $600 billion.'

In [24]:
query = "Compare the two news articles."
answer = rag_chain.invoke(query)
answer

'Based on the provided context, I can compare the two news articles:\n\n1. "8th Pay Commission Update: DA Hike in September? Key Dates & What\'s Next"\n2. "Nashik engineer turns Rs 3,500 investment into a Rs 1.5 crore millet business. Here\'s how"\n\n**Similarities:**\n\n- Both articles are news articles from the same website, moneycontrol.com.\n- Both articles are related to business and finance, with the first article discussing a pay commission update and the second article discussing a successful business venture.\n\n**Differences:**\n\n- The first article is related to government policies and pay commissions, while the second article is related to entrepreneurship and business success.\n- The tone of the first article appears to be informative and neutral, while the second article has a more inspirational and motivational tone.\n- The first article does not have a video link, while the second article has a link to a video titled "Watch more".\n- The first article is not related to

In [25]:
query = "Which company launched Punch iCNG?"
answer = rag_chain.invoke(query)
answer

'Tata Motors launched the Punch iCNG.'

In [33]:
query = "Who is the Prime Minister of India?"
answer = rag_chain.invoke(query)
answer

"I couldn't find the answer in the provided documents."